# 05 · Consolidate, label, prune and balance Docker tickets

Merges everything collected by notebooks 01–04 into one dataset of ~20k `(problem, resolution)` tickets, where every row says
**where it came from, what kind of source it is, and how far to trust it**.

| Step | What happens |
|---|---|
| 1 | Load all sources (notebook 01's Hugging Face output + `raw_sources/*`) into one schema |
| 2 | Mask personal data (emails, public IPs, tokens, usernames in paths, `@mentions`, diagnostic IDs) |
| 3 | Derive features: Docker versions, error strings, exit codes, topics, kind, age |
| 4 | **Prune noise** with named rules; every drop is counted and sampled so you can audit it |
| 5 | Score **trust** (0–1) from source type, votes, maintainer/staff signals, age; assign a tier |
| 6 | Remove near-duplicates (TF-IDF cosine), keeping the most trusted copy |
| 7 | Balance to `TARGET_TOTAL` with per-source caps |

**Type labels:** `official_docs_entry` · `github_maintainer_resolved` · `stackoverflow_accepted` · `stackexchange_accepted` ·
`forum_solved` · `hf_qa_unspecified` · `github_community_resolved` · `reddit_top_comment` · `hf_synthetic_qa`

Missing a source? Skip its collector; this notebook just uses whatever exists.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
TARGET_TOTAL = 20000

# Upper bound per source type. Highest-trust rows win when a source has more than its cap.
SOURCE_CAPS = {
    "stackoverflow_accepted": 9000,
    "github_maintainer_resolved": 5000,
    "forum_solved": 3000,
    "stackexchange_accepted": 2500,
    "github_community_resolved": 500,
    "official_docs_entry": 300,
    "reddit_top_comment": 800,
    "hf_qa_unspecified": 200,
    "hf_synthetic_qa": 800,
}

KEEP_OFFICIAL_DOCS = True             # they duplicate your KB; rows carry overlaps_kb=True either way
HF_SYNTHETIC_KEEP_KINDS = ("troubleshooting", "how_to")   # drop "what is..." concept prompts from Titanium
MIN_TRUST = 0.20                      # floor after scoring
NEAR_DUP_THRESHOLD = 0.90             # TF-IDF cosine similarity on title + problem
USE_DRIVE = True

!pip install -q scikit-learn pandas

In [ ]:
import html
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = Path("/content/drive/MyDrive/customer-support-triage/tickets")
    except ImportError:
        OUT_ROOT = Path("tickets")
else:
    OUT_ROOT = Path("tickets")

RAW_SRC = OUT_ROOT / "raw_sources"
PROCESSED_DIR = OUT_ROOT / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
report = {"run_at_utc": datetime.now(timezone.utc).isoformat(),
          "config": {"target_total": TARGET_TOTAL, "source_caps": SOURCE_CAPS, "min_trust": MIN_TRUST,
                     "near_dup_threshold": NEAR_DUP_THRESHOLD}}
print("Reading from:", OUT_ROOT.resolve())

## 1 · Load everything into one schema

In [ ]:
COLUMNS = ["source_id", "source", "source_type", "url", "license", "created_at", "title", "problem",
           "resolution", "resolution_kind", "problem_score", "resolution_score", "tags", "extra"]


def read_jsonl(path):
    # Notebooks 02-04 append to some files across resumed runs, so a Colab disconnect mid-write can
    # leave one torn line (usually "Unterminated string..."). Skip and report it rather than losing
    # every row in the file to a JSONDecodeError.
    rows, bad = [], []
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    for i, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            bad.append((i, str(exc)))
    if bad:
        print(f"  [warn] {path.name}: skipped {len(bad)} malformed line(s) of {len(lines)}"
              f" (e.g. line {bad[0][0]}: {bad[0][1]})")
    return rows


def load_collected():
    rows = []
    for pattern in ["stackexchange/*.jsonl", "github/*.jsonl", "docs_faq/*.jsonl",
                    "forum/docker_forum.jsonl", "reddit/reddit.jsonl"]:
        for path in sorted(RAW_SRC.glob(pattern)):
            rows += read_jsonl(path)
    return rows


def load_hf():
    # Output of notebook 01. devops-v1 origin is unspecified; Titanium is Llama-3.1-405B synthetic data.
    path = PROCESSED_DIR / "docker_tickets.jsonl"
    if not path.exists():
        return []
    rows = []
    for r in read_jsonl(path):
        synthetic = r["provenance"] == "llm_synthetic"
        if synthetic and r["kind_guess"] not in HF_SYNTHETIC_KEEP_KINDS:
            continue
        rows.append({
            "source_id": f"hf:{r['source']}:{r['source_row']}", "source": r["source"],
            "source_type": "hf_synthetic_qa" if synthetic else "hf_qa_unspecified",
            "url": f"https://huggingface.co/datasets/{r['source']}", "license": "apache-2.0 (dataset card)",
            "created_at": None, "title": r["problem"].split("\n")[0][:200], "problem": r["problem"],
            "resolution": r["resolution"], "resolution_kind": "llm_generated" if synthetic else "curated_answer",
            "problem_score": None, "resolution_score": None, "tags": [], "extra": {"kind_guess": r["kind_guess"]},
        })
    return rows


tickets = pd.DataFrame(load_collected() + load_hf(), columns=COLUMNS)
tickets["tags"] = tickets["tags"].map(lambda t: t if isinstance(t, list) else [])
tickets["extra"] = tickets["extra"].map(lambda e: e if isinstance(e, dict) else {})
if not KEEP_OFFICIAL_DOCS:
    tickets = tickets[tickets["source_type"] != "official_docs_entry"]
report["loaded_by_type"] = tickets["source_type"].value_counts().to_dict()
print(tickets["source_type"].value_counts().to_string(), "\n total:", len(tickets))

## 2 · Clean and mask personal data

Public IP addresses are masked; private ranges (`10.x`, `172.16–31.x`, `192.168.x`, `127.x`, `0.0.0.0`) stay because they are part of
the technical content. Image digests (`sha256:…`) are left alone.

In [ ]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
IPV4_RE = re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b")
MENTION_RE = re.compile(r"(?<![\w@/])@[A-Za-z0-9][A-Za-z0-9-]{0,38}\b(?!/)")
WIN_USER_RE = re.compile(r"(?i)(C:\\Users\\)[^\\\s]+")
NIX_USER_RE = re.compile(r"(/home/|/Users/)[^/\s]+")
DIAG_ID_RE = re.compile(r"(?i)(diagnostic id\W{0,3})[0-9a-f-]{20,}(/\d+)?")
SECRET_PATTERNS = [
    (re.compile(r"\bgh[pousr]_[A-Za-z0-9]{30,}\b"), "<token>"),
    (re.compile(r"\bAKIA[0-9A-Z]{16}\b"), "<aws-key>"),
    (re.compile(r"\bxox[baprs]-[A-Za-z0-9-]{10,}\b"), "<token>"),
    (re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----.*?-----END [A-Z ]*PRIVATE KEY-----", re.S), "<private-key>"),
    (re.compile(r'(?i)("auth"\s*:\s*")[A-Za-z0-9+/=]{20,}'), r"\1<redacted>"),
    (re.compile(r"(?i)(docker login[^\n]*?(?:-p|--password)[ =])\S+"), r"\1<redacted>"),
    (re.compile(r"(?i)\b(password|passwd|token|secret|api[_-]?key)\b(\s*[:=]\s*)(?!<)\S+"), r"\1\2<redacted>"),
]


def mask_ip(match):
    ip = match.group(0)
    a, b = (int(x) for x in ip.split(".")[:2])
    private = a in (0, 10, 127, 255) or (a == 192 and b == 168) or (a == 172 and 16 <= b <= 31) or (a == 169 and b == 254)
    return ip if private else "<ip>"


def scrub(text):
    text = html.unescape(text or "").replace("\r\n", "\n")
    for pattern, replacement in SECRET_PATTERNS:
        text = pattern.sub(replacement, text)
    text = EMAIL_RE.sub("<email>", text)
    text = IPV4_RE.sub(mask_ip, text)
    text = DIAG_ID_RE.sub(r"\1<id>", text)
    text = WIN_USER_RE.sub(r"\1<user>", text)
    text = NIX_USER_RE.sub(r"\1<user>", text)
    text = MENTION_RE.sub("@user", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


for col in ["title", "problem", "resolution"]:
    tickets[col] = tickets[col].map(scrub)
tickets["created_at"] = pd.to_datetime(tickets["created_at"], utc=True, errors="coerce")
print(scrub("Mail me at bob@corp.com, host 8.8.8.8 or 192.168.1.5, path C:\\Users\\bob\\x, ping @alice, password=hunter2"))

## 3 · Derive features

In [ ]:
VERSION_RE = re.compile(
    r"(?i)\b(?:docker(?:[- ](?:desktop|engine|compose|ce|ee))?|compose|buildx)\s*(?:version|v)?\s*[:=]?\s*v?(\d{1,2}\.\d{1,2}(?:\.\d{1,2})?)")
QUOTED_RE = re.compile(r"['\"]([^'\"\n]{12,200})['\"]")
EXIT_RE = re.compile(r"\bexit(?:ed)?(?: with)?(?: code| status)?\s*(\d{1,3})\b", re.I)
ERROR_LINE_RE = re.compile(
    r"(?i)[^\n]{0,60}\b(?:error response from daemon|error:|cannot connect|permission denied|no space left|failed to)\b[^\n]{0,100}")
TROUBLE_RE = re.compile(
    r"\b(error|fail(s|ed|ing|ure)?|not working|cannot|can't|won't|doesn't|isn't|unable|issue|problem|"
    r"troubleshoot|debug|break(s|ing)?|broken|crash(es|ed)?|denied|time(d)? ?out|refused|stuck|exits?|"
    r"exited|hangs?|slow|leak(s|ing)?|invalid|unexpected|corrupt(ed)?|not found|no space|missing|"
    r"conflict|stale|unhealthy)\b", re.I)
QUESTION_RE = re.compile(
    r"^\s*(how|what|why|which|when|where|who|can|could|does|do|is|are|should|would|explain|describe|"
    r"compare|list|write|provide|give|show|create|design|walk)\b", re.I)
HOWTO_RE = re.compile(r"^\s*(how (do|can|to|should)|what (is|are) the (command|steps?|way)|steps? to)", re.I)
TOPIC_PATTERNS = {
    "build": r"dockerfile|docker build|buildx|buildkit|multi-stage|build cache|\blayers?\b",
    "compose": r"compose",
    "engine": r"daemon|dockerd|systemctl|docker\.sock|docker\.pid|/etc/docker|cgroup",
    "networking": r"network|\bports?\b|\bdns\b|bridge|expose|connectivity",
    "storage": r"volume|bind mount|disk|no space|prune|overlay",
    "registry": r"docker hub|dockerhub|registry|\bpull\b|\bpush\b|docker login|rate limit|manifest",
    "desktop": r"docker desktop|\bwsl\b|hyper-v|macos|apple silicon",
    "security": r"permission denied|rootless|privileged|seccomp|apparmor|vulnerab|user namespace",
    "runtime": r"exits?|stops?|crash|\boom\b|memory|restart|healthcheck|exit code",
    "swarm": r"swarm|stack deploy",
}
TOPIC_RES = {t: re.compile(p, re.I) for t, p in TOPIC_PATTERNS.items()}
FENCE_RE = re.compile(r"```.*?```", re.S)
URL_RE = re.compile(r"https?://\S+")
NOW = pd.Timestamp.now(tz="UTC")


def guess_kind(title, problem, errors):
    if errors or not QUESTION_RE.match(title or problem) or TROUBLE_RE.search(title):
        return "troubleshooting"
    if HOWTO_RE.match(title or problem):
        return "how_to"
    return "concept"


def enrich(row):
    head = f"{row.title}\n{row.problem}"
    errors = [m.group(1).strip() for m in QUOTED_RE.finditer(head)]
    errors += [m.group(0).strip()[:200] for m in ERROR_LINE_RE.finditer(row.problem)][:3]
    return pd.Series({
        "docker_versions": list(dict.fromkeys(VERSION_RE.findall(head + "\n" + row.resolution)))[:3],
        "error_strings": list(dict.fromkeys(errors))[:5],
        "exit_codes": sorted(set(EXIT_RE.findall(head))),
        "topics": [t for t, rx in TOPIC_RES.items() if rx.search(head)],
        "kind_guess": guess_kind(row.title, row.problem, errors),
        "has_code": "```" in row.problem or "```" in row.resolution,
        "problem_words": len(row.problem.split()),
        "resolution_words": len(row.resolution.split()),
        "link_ratio": sum(len(u) for u in URL_RE.findall(row.resolution)) / max(1, len(row.resolution)),
    })


tickets = pd.concat([tickets.reset_index(drop=True), tickets.reset_index(drop=True).apply(enrich, axis=1)], axis=1)
tickets["age_years"] = ((NOW - tickets["created_at"]).dt.days / 365.25).round(1)
print(tickets.groupby(["source_type", "kind_guess"]).size().unstack(fill_value=0).to_string())

## 4 · Prune noise

Each rule has a name. The first rule a row fails is its `prune_reason`; counts go into the report and a sample of dropped rows is
saved to `pruned_sample.jsonl` so you can check the rules are not throwing away good data.

In [ ]:
DOCKER_TERM_RE = re.compile(r"docker|moby|containerd|compose|buildx|buildkit", re.I)
BOILERPLATE_RE = re.compile(
    r"(?i)(closing( this)?( issue)?|closed due to|stale|inactiv\w*|duplicate of|please (open|file|create) a new issue|"
    r"this issue has been automatically|\+1|me too|same (here|issue)|any update|\bbump\b|thanks?[ !.]*$)")
FEATURE_TITLE_RE = re.compile(r"(?i)^\W*(\[?feature|feature request|proposal|enhancement|suggestion)")
# These sources are Docker-specific by where they came from, so a missing "docker" keyword is not a reason to drop them.
DOCKER_BY_CONSTRUCTION = {"official_docs_entry", "forum_solved", "github_maintainer_resolved", "github_community_resolved"}


EN_STOPWORDS = set(
    "the and is to of in it that for on with this you are not be as at by from or an if can but have was your has do does "
    "when what which will would there they them their so than then into out about no yes we i my me use used using "
    "should could may also more only any one all other how why".split())


def ascii_letter_share(text):
    letters = [c for c in text if c.isalpha()]
    return sum(c.isascii() for c in letters) / len(letters) if letters else 1.0


def looks_english(text, min_words=20, threshold=0.06):
    # Catches Latin-script languages (Spanish, Portuguese, German...) that the ASCII check cannot.
    words = re.findall(r"[a-zA-Z']+", FENCE_RE.sub(" ", text).lower())
    if len(words) < min_words:
        return True
    return sum(w in EN_STOPWORDS for w in words) / len(words) >= threshold


def code_fraction(text):
    return sum(len(b) for b in FENCE_RE.findall(text)) / max(1, len(text))


def prune_reason(row):
    resolution_core = URL_RE.sub("", BOILERPLATE_RE.sub("", row.resolution)).strip()
    text_all = f"{row.title} {row.problem} {row.resolution} {' '.join(row.tags)}"
    if len(row.problem) < 15 or len(row.resolution) < 40:
        return "too_short"
    if row.problem_words > 2000 or row.resolution_words > 3000:
        return "too_long"
    if row.source_type not in DOCKER_BY_CONSTRUCTION and not DOCKER_TERM_RE.search(text_all):
        return "not_docker"
    if ascii_letter_share(row.problem + row.resolution) < 0.90 or not looks_english(row.problem + " " + row.resolution):
        return "non_english"
    if len(resolution_core) < 30:
        return "boilerplate_or_link_only_resolution"
    if FEATURE_TITLE_RE.match(row.title):
        return "feature_request"
    no_descriptive_title = len(row.title.split()) < 5
    if no_descriptive_title and code_fraction(row.problem) > 0.85 and len(FENCE_RE.sub("", row.problem).split()) < 15:
        return "log_dump_problem"
    return None


tickets["prune_reason"] = tickets.apply(prune_reason, axis=1)
report["pruned_by_rule"] = (tickets.dropna(subset=["prune_reason"])
                            .groupby(["prune_reason", "source_type"]).size().unstack(fill_value=0).to_dict("index"))
pruned = tickets[tickets["prune_reason"].notna()]
pruned.groupby("prune_reason").head(15)[["prune_reason", "source_type", "title", "url"]].to_json(
    PROCESSED_DIR / "pruned_sample.jsonl", orient="records", lines=True, force_ascii=False)

tickets = tickets[tickets["prune_reason"].isna()].drop(columns="prune_reason").reset_index(drop=True)
print(pruned["prune_reason"].value_counts().to_string(), "\n kept:", len(tickets))

## 5 · Trust score and tier

Transparent rules, not a model: a base score per source type, adjusted by evidence of quality. Change `BASE_TRUST` to your own
judgement. Each row lists the reasons behind its score in `trust_reasons`.

In [ ]:
BASE_TRUST = {
    "official_docs_entry": 0.90, "github_maintainer_resolved": 0.75, "stackoverflow_accepted": 0.70,
    "stackexchange_accepted": 0.65, "forum_solved": 0.60, "hf_qa_unspecified": 0.45,
    "github_community_resolved": 0.45, "reddit_top_comment": 0.35, "hf_synthetic_qa": 0.25,
}


def score_trust(row):
    score, reasons = BASE_TRUST.get(row.source_type, 0.30), [f"base {row.source_type}"]

    votes = row.resolution_score
    if pd.notna(votes):
        for threshold, bump in ((50, 0.15), (10, 0.10), (3, 0.05)):
            if votes >= threshold:
                score += bump
                reasons.append(f"answer score {int(votes)}")
                break
        else:
            if votes <= 0:
                score -= 0.10
                reasons.append("answer score <= 0")

    if row.extra.get("closed_by_pr"):
        score += 0.10
        reasons.append("fix linked to a PR")
    if row.extra.get("staff_answer"):
        score += 0.10
        reasons.append("answered by forum staff")
    if row.resolution_kind == "op_confirmed_comment":
        score += 0.10
        reasons.append("poster confirmed it worked")
    if pd.notna(row.age_years) and row.age_years > 6:
        score -= min(0.15, 0.05 * (row.age_years - 6))
        reasons.append(f"old post ({row.age_years}y, Docker has changed)")
    if row.link_ratio > 0.5:
        score -= 0.10
        reasons.append("answer is mostly links")
    if row.resolution_words < 25 and row.source_type != "official_docs_entry":
        score -= 0.05
        reasons.append("very short answer")
    return pd.Series({"trust_score": round(min(1.0, max(0.0, score)), 2), "trust_reasons": reasons})


tickets = pd.concat([tickets, tickets.apply(score_trust, axis=1)], axis=1)
tickets["trust_tier"] = pd.cut(tickets["trust_score"], [-0.01, 0.45, 0.70, 1.01], labels=["low", "medium", "high"], right=False)
tickets["is_synthetic"] = tickets["source_type"].eq("hf_synthetic_qa")
tickets["overlaps_kb"] = tickets["source_type"].eq("official_docs_entry")

before = len(tickets)
tickets = tickets[tickets["trust_score"] >= MIN_TRUST].reset_index(drop=True)
report["dropped_below_min_trust"] = before - len(tickets)
print(tickets.groupby("source_type")["trust_score"].agg(["count", "mean"]).round(2).to_string())

## 6 · Remove near-duplicates

Exact copies and near-copies (cosine ≥ `NEAR_DUP_THRESHOLD` on title + start of the problem) are collapsed, keeping the row with the
highest trust. Comparing in blocks keeps memory small even at ~30k rows.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


def drop_near_duplicates(df, threshold, block=1000):
    order = np.lexsort((-df["resolution_score"].fillna(0).values, -df["trust_score"].values))
    df = df.iloc[order].reset_index(drop=True)          # best rows first
    texts = (df["title"] + " " + df["problem"].str[:1500]).tolist()
    matrix = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True,
                             stop_words="english", dtype=np.float32).fit_transform(texts)
    keep = np.ones(len(df), dtype=bool)
    for start in range(0, len(df), block):
        sims = (matrix[start:start + block] @ matrix.T).toarray()
        for local in range(sims.shape[0]):
            i = start + local
            if not keep[i]:
                continue
            row = sims[local]
            row[:i + 1] = 0                             # only compare against lower-priority rows
            keep[np.where(row >= threshold)[0]] = False
    return df[keep].reset_index(drop=True)


before = len(tickets)
tickets = drop_near_duplicates(tickets, NEAR_DUP_THRESHOLD)
report["dropped_near_duplicates"] = before - len(tickets)
print("near-duplicates removed:", report["dropped_near_duplicates"], "| remaining:", len(tickets))

## 7 · Balance to the target size

In [ ]:
ranked = tickets.sort_values(["trust_score", "resolution_score"], ascending=False, na_position="last")
within_type_rank = ranked.groupby("source_type").cumcount()
capped = ranked[within_type_rank < ranked["source_type"].map(SOURCE_CAPS).fillna(500)]
report["dropped_by_source_cap"] = len(tickets) - len(capped)

if len(capped) > TARGET_TOTAL:
    capped = capped.sort_values(["trust_score", "resolution_score"], ascending=False, na_position="last").head(TARGET_TOTAL)
report["shortfall_vs_target"] = max(0, TARGET_TOTAL - len(capped))
final = capped.reset_index(drop=True)
print("final rows:", len(final), "| shortfall vs target:", report["shortfall_vs_target"])

## 8 · Save

In [ ]:
import hashlib

final["ticket_id"] = ["T-" + hashlib.sha1(s.encode()).hexdigest()[:12] for s in final["source_id"]]
final["created_at"] = final["created_at"].dt.strftime("%Y-%m-%d")
final["trust_tier"] = final["trust_tier"].astype(str)

out_columns = ["ticket_id", "source_id", "source", "source_type", "trust_tier", "trust_score", "trust_reasons",
               "is_synthetic", "overlaps_kb", "license", "url", "created_at", "age_years", "title", "problem",
               "resolution", "resolution_kind", "problem_score", "resolution_score", "kind_guess", "topics", "tags",
               "docker_versions", "error_strings", "exit_codes", "has_code", "problem_words", "resolution_words"]
final = final[out_columns]

final.to_json(PROCESSED_DIR / "docker_tickets_v2.jsonl", orient="records", lines=True, force_ascii=False)
csv_frame = final.copy()
for col in ["trust_reasons", "topics", "tags", "docker_versions", "error_strings", "exit_codes"]:
    csv_frame[col] = csv_frame[col].map(json.dumps)
csv_frame.to_csv(PROCESSED_DIR / "docker_tickets_v2.csv", index=False)

report.update({
    "final_count": len(final),
    "final_by_source_type": final["source_type"].value_counts().to_dict(),
    "final_by_trust_tier": final["trust_tier"].value_counts().to_dict(),
    "final_by_kind": final["kind_guess"].value_counts().to_dict(),
    "final_by_year": final["created_at"].str[:4].value_counts().sort_index().to_dict(),
    "share_synthetic": round(float(final["is_synthetic"].mean()), 3),
})
(PROCESSED_DIR / "docker_tickets_v2_report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print(json.dumps({k: report[k] for k in ["final_count", "final_by_source_type", "final_by_trust_tier", "final_by_kind"]}, indent=2))

## 9 · Spot-check

In [ ]:
for source_type, group in final.groupby("source_type"):
    row = group.sample(1, random_state=1).iloc[0]
    print(f"\n[{source_type} | trust {row.trust_score} {row.trust_tier}] {row.title[:110]}")
    print("   why:", "; ".join(row.trust_reasons))
    print("   ->", row.resolution[:150].replace("\n", " "))

# Optional: download.
try:
    from google.colab import files
    for name in ["docker_tickets_v2.jsonl", "docker_tickets_v2_report.json"]:
        files.download(str(PROCESSED_DIR / name))
except ImportError:
    pass